In [1]:
import pandas as pd

feature = pd.read_csv("feature_final_dataset.csv")
whois = pd.read_csv("~/Bitirme_Projesi_Dataset_Olusturma/Feature/whois_feature_merged.csv")

print("FEATURE kolonları:", feature.columns.tolist())
print("WHOIS kolonları:", whois.columns.tolist())


FEATURE kolonları: ['current_url', 'label', 'url_length', 'domain_length', 'path_length', 'query_length', 'entropy_url', 'entropy_domain', 'entropy_path', 'num_digits', 'digit_ratio', 'num_slash', 'num_dot', 'num_hyphen', 'num_underscore', 'num_special', 'num_params', 'subdomain_count', 'has_https_token', 'has_at_symbol', 'has_double_slash', 'starts_with_ip', 'contains_port_number', 'path_segments_count', 'path_has_encoded_chars', 'query_entropy', 'query_has_base64', 'query_key_count', 'query_value_length_avg', 'special_ratio', 'uppercase_ratio', 'lowercase_ratio', 'success', 'dns_resolves', 'has_mx_record', 'has_txt_record', 'has_ns_record', 'ttl_value', 'ip_count', 'cname_count', 'resolves_to_private_ip']
WHOIS kolonları: ['domain', 'whois_success', 'domain_age_days', 'expiration_days', 'creation_year', 'domain_is_recent', 'domain_registered_before_2020', 'registrar_valid', 'name_servers_count', 'is_privacy_protected']


In [2]:
feature["domain"] = (
    feature["current_url"]
    .str.extract(r"https?://([^/]+)", expand=False)
    .str.lower()
    .str.strip()
    .str.replace(r"^www\.", "", regex=True)
    .str.replace(r"\.$", "", regex=True)
)


In [3]:
import pandas as pd

# ============================
# 1) Dosyaları yükle
# ============================
feature = pd.read_csv("feature_final_dataset.csv")
whois = pd.read_csv("~/Bitirme_Projesi_Dataset_Olusturma/Feature/whois_feature_merged.csv")

print("Feature rows:", len(feature))
print("WHOIS rows:", len(whois))


# ============================
# 2) Feature dataset → domain çıkar
# ============================
feature["domain"] = (
    feature["current_url"]
    .str.extract(r"https?://([^/]+)", expand=False)  # URL’den domain'i al
    .str.lower()
    .str.strip()
    .str.replace(r"^www\.", "", regex=True)
    .str.replace(r"\.$", "", regex=True)
)

print("Domain örnek (feature):")
print(feature["domain"].head())


# ============================
# 3) WHOIS → domain normalize
# ============================
whois["domain"] = (
    whois["domain"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"^www\.", "", regex=True)
    .str.replace(r"\.$", "", regex=True)
)

print("Domain örnek (whois):")
print(whois["domain"].head())


# ============================
# 4) WHOIS duplicate domain temizliği
# ============================
before = len(whois)
whois = whois.drop_duplicates(subset="domain", keep="last")
after = len(whois)

print(f"WHOIS duplicate silindi: {before - after} kayıt")


# ============================
# 5) DOĞRU MERGE
# ============================
merged = feature.merge(
    whois,
    on="domain",
    how="left"        # Çok kritik! Satır sayısı sabit kalır
)


# ============================
# 6) Sonuç kontrol
# ============================
print("Merged rows:", len(merged))

if len(merged) == len(feature):
    print("✔ Satır sayısı değişmedi! Merge başarılı.")
else:
    print("❌ Satır sayısı değişti! Bir sorun var.")


# ============================
# 7) Final dataset kaydet
# ============================
merged.to_csv("feature_with_whois_final2222.csv", index=False)
print("Final CSV kaydedildi: feature_with_whois_final2222.csv")


Feature rows: 579926
WHOIS rows: 133969
Domain örnek (feature):
0         5ch.net
1         5ch.net
2      abcvg.info
3      abcvg.info
4    adsafety.net
Name: domain, dtype: object
Domain örnek (whois):
0     adsafety.net
1    asilmedia.org
2       akerala.in
3          5ch.net
4        railsc.ru
Name: domain, dtype: object
WHOIS duplicate silindi: 4 kayıt
Merged rows: 579926
✔ Satır sayısı değişmedi! Merge başarılı.
Final CSV kaydedildi: feature_with_whois_final2222.csv


In [4]:
import pandas as pd
import tldextract

# ============================
# 1) Feature dataset
# ============================
feature_path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/feature_final_dataset.csv"
feature = pd.read_csv(feature_path)

# TLDExtract domain extraction
def get_domain(url):
    ext = tldextract.extract(url)
    return ext.registered_domain  # example.com

feature["domain"] = feature["current_url"].apply(get_domain)
feature["domain"] = feature["domain"].str.lower().str.strip()

print("Feature rows:", len(feature))
print("Unique domains (feature):", feature["domain"].nunique())


# ============================
# 2) WHOIS dataset
# ============================
whois_path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/whois_feature_merged.csv"
whois = pd.read_csv(whois_path)

whois["domain"] = (
    whois["domain"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Duplicate WHOIS domain temizliği
before = len(whois)
whois = whois.drop_duplicates(subset="domain", keep="last")
after = len(whois)

print("WHOIS rows:", after)
print("Silinen duplicate WHOIS:", before - after)
print("Unique domains (whois):", whois["domain"].nunique())


# ============================
# 3) Merge
# ============================
merged = feature.merge(
    whois,
    on="domain",
    how="left"   # Sadece feature satırlarını korur
)

print("Merged rows:", len(merged))


# ============================
# 4) Kontrol
# ============================
if len(merged) == len(feature):
    print("✔ Merge başarılı! Satır sayısı değişmedi.")
else:
    print("❌ Merge hatası! Satır sayısı değişti.")


# ============================
# 5) Final CSV kaydet
# ============================
out_path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_feature_with_whois_sonolsun.csv"
merged.to_csv(out_path, index=False)

print("Final dataset kaydedildi:", out_path)


C:\Users\elifo\AppData\Local\Temp\ipykernel_6060\707539322.py:13: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  return ext.registered_domain  # example.com


Feature rows: 579926
Unique domains (feature): 133965
WHOIS rows: 133964
Silinen duplicate WHOIS: 0
Unique domains (whois): 133964
Merged rows: 579926
✔ Merge başarılı! Satır sayısı değişmedi.
Final dataset kaydedildi: ~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_feature_with_whois_sonolsun.csv


In [5]:
import pandas as pd

df = pd.read_csv("~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois_sonolsun.csv")
print("Toplam satır:", len(df))


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\elifo/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois_sonolsun.csv'

In [7]:
import pandas as pd
import tldextract

# ============================
# 1) LEXICAL + DNS DATASET
# ============================
lexical_path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/Untitled Folder/lexical+Dns.csv"
lexical = pd.read_csv(lexical_path)

print("Lexical rows:", len(lexical))

# Domain extraction (tldextract)
def extract_domain(url):
    ext = tldextract.extract(url)
    return ext.registered_domain  # example.com

lexical["domain"] = lexical["url"].apply(extract_domain)
lexical["domain"] = (
    lexical["domain"]
    .str.lower()
    .str.strip()
    .str.replace(r"\.$", "", regex=True)
)

print("Unique domains in lexical:", lexical["domain"].nunique())


# ============================
# 2) WHOIS DATASET
# ============================
whois_path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/whois_feature_merged.csv"
whois = pd.read_csv(whois_path)

# Normalize WHOIS domain
whois["domain"] = (
    whois["domain"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\.$", "", regex=True)
)

# Remove duplicate domain entries (safety)
before = len(whois)
whois = whois.drop_duplicates(subset="domain", keep="last")
after = len(whois)

print("WHOIS original rows:", before)
print("WHOIS cleaned rows :", after)
print("Unique domains in WHOIS:", whois["domain"].nunique())


# ============================
# 3) MERGE (LEFT JOIN)
# ============================
merged = lexical.merge(
    whois,
    on="domain",
    how="left"
)

print("Merged rows:", len(merged))

# Check correctness
if len(merged) == len(lexical):
    print("✔ Merge successful! Row count unchanged.")
else:
    print("❌ Merge error! Row count changed.")


# ============================
# 4) SAVE FINAL MERGED FILE
# ============================
output_path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois.csv"
merged.to_csv(output_path, index=False)

print("Final merged dataset saved to:", output_path)


Lexical rows: 579934


C:\Users\elifo\AppData\Local\Temp\ipykernel_6060\2845135683.py:15: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  return ext.registered_domain  # example.com


Unique domains in lexical: 133965
WHOIS original rows: 133964
WHOIS cleaned rows : 133964
Unique domains in WHOIS: 133964
Merged rows: 579934
✔ Merge successful! Row count unchanged.
Final merged dataset saved to: ~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois.csv


In [8]:
import pandas as pd

df = pd.read_csv("~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois.csv")

print("Toplam satır sayısı:", len(df))

# NaN içeren kolonları bul
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]

print("\n=== NaN Olan Kolonlar ve NaN Sayıları ===")
print(nan_cols)


Toplam satır sayısı: 579934

=== NaN Olan Kolonlar ve NaN Sayıları ===
domain                           6608
whois_success                    6845
domain_age_days                  6845
expiration_days                  6845
creation_year                    6845
domain_is_recent                 6845
domain_registered_before_2020    6845
registrar_valid                  6845
name_servers_count               6845
is_privacy_protected             6845
dtype: int64


In [9]:
import pandas as pd

# Dosyayı yükle
df = pd.read_csv("~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois.csv")

# 1) WHOIS bilgisi eksik mi? → Binary flag
whois_cols = [
    "whois_success", "domain_age_days", "expiration_days", "creation_year",
    "domain_is_recent", "domain_registered_before_2020", "registrar_valid",
    "name_servers_count", "is_privacy_protected"
]

df["whois_missing"] = df[whois_cols].isna().any(axis=1).astype(int)

# 2) Tüm NaN değerlerini 0 ile doldur
df.fillna(0, inplace=True)

# Final dataset olarak kaydet
df.to_csv(
    "~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois_cleaned.csv",
    index=False
)

print("✔ Missing flag eklendi ve NaN değerler temizlendi.")


✔ Missing flag eklendi ve NaN değerler temizlendi.


In [10]:
import pandas as pd

path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois_cleaned.csv"

# Dosyayı yükle
df = pd.read_csv(path)

print("Silme öncesi kolon sayısı:", len(df.columns))
print("Kolonlar:", df.columns.tolist())

# DOMAIN kolonunu sil
df = df.drop(columns=["domain"], errors="ignore")

print("\nSilme sonrası kolon sayısı:", len(df.columns))
print("Kalan kolonlar:", df.columns.tolist())

# Aynı dosya üzerine kaydet (kalıcı temizlik)
df.to_csv(path, index=False)

print("\n✔ 'domain' kolonu kalıcı olarak silindi ve dosya güncellendi!")


Silme öncesi kolon sayısı: 78
Kolonlar: ['url', 'label', 'url_length', 'domain_length', 'hostname_length', 'path_length', 'first_dir_length', 'tld_length', 'tld_length_domain', 'url_depth', 'query_length', 'path_segments_count', 'num_digits', 'num_letters', 'num_special_chars', 'num_dots', 'num_hyphens', 'num_at', 'num_percent', 'num_equals', 'num_question', 'num_ampersand', 'num_hash', 'num_underscore', 'num_special', 'num_slash', 'num_params', 'entropy_url', 'entropy_hostname', 'entropy_domain', 'entropy_path', 'query_entropy', 'ratio_digits', 'ratio_letters', 'ratio_special_chars', 'uppercase_ratio', 'lowercase_ratio', 'is_ip_address', 'starts_with_ip', 'is_suspicious_tld', 'uses_https', 'has_www', 'num_at.1', 'unusual_double_slash', 'multiple_http', 'contains_port_number', 'path_has_encoded_chars', 'query_has_base64', 'contains_login', 'contains_secure', 'contains_verify', 'contains_account', 'contains_update', 'contains_bank', 'contains_cloud', 'contains_brand', 'query_key_count',

In [11]:
import pandas as pd

path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois_cleaned.csv"
df = pd.read_csv(path)

# URL kolonunuz hangi isimdeyse otomatik algılayalım
url_col = "url" if "url" in df.columns else "current_url"

print("Kullanılan URL kolonu:", url_col)

# Duplicate kontrol
duplicate_count = df[url_col].duplicated().sum()

print("\n=== URL Bazlı Duplicate Sayısı ===")
print(duplicate_count)

if duplicate_count > 0:
    print("\n=== Duplicate Olan İlk 10 URL ===")
    print(df[df[url_col].duplicated()][url_col].head(10))
else:
    print("\n✔ Hiç duplicate URL yok! Dataset tamamen temiz.")


Kullanılan URL kolonu: url

=== URL Bazlı Duplicate Sayısı ===
14

=== Duplicate Olan İlk 10 URL ===
449572      https://serviciodecorreo.es/
449573      https://serviciodecorreo.es/
449574      https://serviciodecorreo.es/
449575      https://serviciodecorreo.es/
449576      https://serviciodecorreo.es/
449577      https://serviciodecorreo.es/
449578      https://serviciodecorreo.es/
553762    https://webmail-seguro.com.br/
553763    https://webmail-seguro.com.br/
553764    https://webmail-seguro.com.br/
Name: url, dtype: object


In [12]:
import pandas as pd

path = "~/Bitirme_Projesi_Dataset_Olusturma/Feature/final_lexical_dns_whois_cleaned.csv"

# Dosyayı yükle
df = pd.read_csv(path)

# URL kolonunu belirle
url_col = "url" if "url" in df.columns else "current_url"

# Duplicate sayısı
dup_count = df[url_col].duplicated().sum()
print("Silinecek duplicate URL sayısı:", dup_count)

# Duplicate’leri sil (ilk görüleni tut, diğerlerini at)
df_clean = df.drop_duplicates(subset=url_col, keep="first")

print("Yeni satır sayısı:", len(df_clean))

# Kalıcı olarak kaydet
df_clean.to_csv(path, index=False)

print("\n✔ Duplicate URL’ler başarıyla temizlendi ve dosya güncellendi!")


Silinecek duplicate URL sayısı: 14
Yeni satır sayısı: 579920

✔ Duplicate URL’ler başarıyla temizlendi ve dosya güncellendi!


In [14]:
import pandas as pd

df = pd.read_csv("~/Bitirme_Projesi_Dataset_Olusturma/Feature/onemli/final_lexical_dns_whois_cleaned.csv")

print("Toplam satır sayısı:", len(df))

# NaN içeren kolonları bul
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]

print("\n=== NaN Olan Kolonlar ve NaN Sayıları ===")
print(nan_cols)

Toplam satır sayısı: 579920

=== NaN Olan Kolonlar ve NaN Sayıları ===
Series([], dtype: int64)
